# 05. Resolution and Occlusion Sensitivity

Run a controlled synthetic sensitivity study. This first version isolates scan degradation by using true fragment labels after degradation, then estimating visible fragment volumes from partial point clusters. Segmentation error is evaluated separately in notebooks 03 and 04.

In [ ]:
from pathlib import Path
import sys
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from scanning.virtual_lidar import viewpoint_filter
from scanning.scan_noise import reduce_density, add_range_noise
from scanning.occlusion_model import keep_nearest_by_angular_bin
from fragmentation.volume_estimation import estimate_cluster_volumes_convex_hull
from fragmentation.psd import equivalent_spherical_diameter, cumulative_psd, percentile_size
from fragmentation.p80 import p80_error_mm, p80_relative_error_pct

SCAN_DIR = ROOT / 'data' / 'scanned_pointclouds'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
for path in [TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

dense = np.load(SCAN_DIR / 'synthetic_rockpile_full_labelled_pointcloud.npz')
dense_points = dense['points_xyz']
dense_labels = dense['fragment_ids']
gt_summary = pd.read_csv(TABLE_DIR / 'ground_truth_summary.csv').iloc[0]
gt_p80 = float(gt_summary['P80_mm'])

PARAMS = {
    'viewpoint_xyz': np.array([2.4, -2.2, 1.35]),
    'look_at_xyz': np.array([0.0, 0.0, 0.28]),
    'field_of_view_deg': 88.0,
    'max_range_m': 5.0,
    'max_points_per_case': 18_000,
    'random_seed': 2026,
}

density_levels = [1.0, 0.55, 0.30]
noise_levels_m = [0.0, 0.004, 0.010]
occlusion_bins_deg = [0.10, 0.18, 0.35]
design = list(itertools.product(density_levels, noise_levels_m, occlusion_bins_deg))
len(design)

In [ ]:
rows = []
for case_id, (density_keep, noise_sigma, occlusion_bin) in enumerate(design):
    seed = PARAMS['random_seed'] + case_id * 37
    view = viewpoint_filter(
        dense_points,
        dense_labels,
        viewpoint_xyz=PARAMS['viewpoint_xyz'],
        max_range_m=PARAMS['max_range_m'],
        field_of_view_deg=PARAMS['field_of_view_deg'],
        look_at_xyz=PARAMS['look_at_xyz'],
    )
    occluded_points, occluded_labels = keep_nearest_by_angular_bin(
        view.points_xyz,
        view.fragment_ids,
        viewpoint_xyz=PARAMS['viewpoint_xyz'],
        angular_resolution_deg=occlusion_bin,
    )
    degraded_points, degraded_labels = reduce_density(occluded_points, occluded_labels, density_keep, random_seed=seed)
    if len(degraded_points) > PARAMS['max_points_per_case']:
        degraded_points, degraded_labels = reduce_density(
            degraded_points,
            degraded_labels,
            PARAMS['max_points_per_case'] / len(degraded_points),
            random_seed=seed + 1,
        )
    degraded_points = add_range_noise(degraded_points, PARAMS['viewpoint_xyz'], noise_sigma, random_seed=seed + 2)

    volumes = estimate_cluster_volumes_convex_hull(degraded_points, degraded_labels)
    volumes = volumes[(volumes['n_points'] >= 20) & (volumes['estimated_volume_m3'] > 1e-8)].copy()
    if len(volumes) >= 3:
        diameters = equivalent_spherical_diameter(volumes['estimated_volume_m3'].to_numpy())
        psd = cumulative_psd(diameters, volumes['estimated_volume_m3'].to_numpy())
        estimated_p80 = percentile_size(psd, 80.0)
        error_mm = p80_error_mm(estimated_p80, gt_p80)
        error_pct = p80_relative_error_pct(estimated_p80, gt_p80)
    else:
        estimated_p80 = np.nan
        error_mm = np.nan
        error_pct = np.nan

    rows.append({
        'case_id': case_id,
        'density_keep_fraction': density_keep,
        'noise_sigma_m': noise_sigma,
        'occlusion_bin_deg': occlusion_bin,
        'n_points': int(len(degraded_points)),
        'n_visible_fragments': int(len(np.unique(degraded_labels))),
        'n_volume_estimates': int(len(volumes)),
        'ground_truth_P80_mm': gt_p80,
        'estimated_P80_mm': estimated_p80,
        'P80_error_mm': error_mm,
        'P80_relative_error_pct': error_pct,
    })

sensitivity = pd.DataFrame(rows)
sensitivity_path = TABLE_DIR / 'p80_error_sensitivity.csv'
sensitivity.to_csv(sensitivity_path, index=False)
sensitivity.head()

In [ ]:
summary = sensitivity.groupby(['density_keep_fraction', 'occlusion_bin_deg'], as_index=False)['P80_relative_error_pct'].mean()
pivot = summary.pivot(index='density_keep_fraction', columns='occlusion_bin_deg', values='P80_relative_error_pct')

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(pivot.to_numpy(), cmap='coolwarm', aspect='auto')
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_xticklabels([f'{x:.2f}' for x in pivot.columns])
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels([f'{x:.2f}' for x in pivot.index])
ax.set_xlabel('Occlusion angular bin [deg]')
ax.set_ylabel('Density keep fraction')
ax.set_title('Mean P80 relative error [%]')
for y in range(pivot.shape[0]):
    for x in range(pivot.shape[1]):
        value = pivot.to_numpy()[y, x]
        ax.text(x, y, f'{value:.1f}', ha='center', va='center', color='black')
fig.colorbar(image, ax=ax, label='P80 relative error [%]')
fig.tight_layout()
figure_path = FIGURE_DIR / 'p80_sensitivity_heatmap.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)